In [6]:
import os
import sys
from tqdm import tqdm
import numpy as np
import pandas as pd
import scipy.sparse as sps
from scipy import sparse
import scipy as sp
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
# from nilearn import image, surface, plotting, datasets
from tqdm import tqdm
from matplotlib import font_manager
font_manager.fontManager.addfont("/n02dat01/users/lchai/anaconda3/envs/Nm/lib/python3.8/site-packages/matplotlib/mpl-data/fonts/ttf/arial.ttf")
plt.rcParams["font.sans-serif"] = "Arial" 

import warnings
warnings.filterwarnings('ignore')

In [7]:
_ = np.array([0, 2, 12, 14, 16, 18, 20, 22, 24, 26, 29, 31, 33, 35, 37, 39, 41, 43, 46, 48, 50, 52, 54, 56, 58, 60, 62, 64, 66, 68, 70, 4, 5, 6, 7, 8, 9, 10, 11, 28, 45])
# read the fiber name
l_idx = [1,3,13,15,17,19,25,27,30,32,36,38,40,42,44,47,49,51,53,55,57,59,61,63,65,67,69,71]
r_idx = [2,4,14,16,18,24,26,28,31,35,37,39,41,43,45,48,50,52,54,56,58,60,62,64,66,68,70,72]
m_idx = [4,5,6,7,8,9,10,11]
l_idx = np.array(l_idx)
r_idx = np.array(r_idx)
m_idx = np.array(m_idx)
l_idx = l_idx-1
r_idx = r_idx-1
label_f = open('/n02dat01/users/dyli/Grad_data/support_data/fiber_name_ori_nonum_nohemi.txt', 'r')
label_name = label_f.readlines()
label_name = [' '.join([i.strip() for i in price.strip().split('\n')]) for price in label_name]
label_name_lm = [label_name[l_idx[i]] for i in range(len(l_idx))] + [label_name[m_idx[i]] for i in range(len(m_idx))]
print(f'the number of fiber: {len(label_name_lm)}')

new_fiber_idx = []
for fi,ff in enumerate(_):
    if ff in list(l_idx)+list(m_idx): new_fiber_idx.append(fi)
new_fiber_idx = np.array(new_fiber_idx)
print(new_fiber_idx.shape)

the number of fiber: 36
(36,)


# rotation for eigenmodes: doi: https://doi.org/10.1101/2024.02.07.579070

In [ ]:
! cd eigenstrapping
! python3 -m pip install ./eigenstrapping

In [3]:
# remove the medial wall
dirc_L = '/n02dat01/users/dyli/Atlas/metric_index_L.txt'
select_ind_L = np.loadtxt( dirc_L ).astype(int)
dirc_R = '/n02dat01/users/dyli/Atlas/metric_index_R.txt'
select_ind_R = np.loadtxt( dirc_R ).astype(int)
# read the group MODE
x = np.loadtxt('/n01dat01/dyli/multi/support_code/BrainEigenmodes/data/template_eigenmodes/fsLR_32k_white-lh_emode_200.txt') # (32492, 200)
x = x[select_ind_L,:]

In [4]:
def read_real_FP(sub):
    # read the real fingerprint
    _ = sps.load_npz(f'/n04dat01/atlas_group/lma/HCP_S1200_individual_MSM_atlas/{sub}/{sub}_L_probtrackx_omatrix2/finger_print_fiber_MSMALL.npz')
    _ = _.toarray()
    # check the all-zero row
    for ii in range(_.shape[0]):
        if len(np.unique(_[ii,:])) == 1: _[ii,:]=_[ii-1,:]
    for ii in range(_.shape[0]):
        if len(np.unique(_[ii,:])) == 1: sys.exit()
    # normalization
    fingerprint = np.array([_[:,i]/np.sum(_, axis=1) for i in range(_.shape[1])]).T
    # choose the left fibers and the cc fibers
    fingerprint = fingerprint[:, np.array(list(l_idx) + list(m_idx))]
    assert fingerprint.shape[0]==29696 and fingerprint.shape[1]==int(len(l_idx)+len(m_idx))
    # thr
    fingerprint[fingerprint<0.05] = 0

    return fingerprint

def read_constracted_FP(sub):
    para = np.load(f'/n01dat01/dyli/multi/HCP_1200/{sub}/FP_{sub}_predict_by_200_group_whitemode_deve-mode_thr05_para_L.npy')[:,new_fiber_idx]
    return np.squeeze(np.dot(x,para))

In [5]:
from eigenstrapping import SurfaceEigenstrapping

sub = '100307'
data_real = read_real_FP(sub)
data_reconstracted = read_constracted_FP(sub)

sur = np.zeros((1001, len(label_name_lm)))
for ff in range(len(label_name_lm)):
    print(label_name_lm[ff])
    sur[0,ff] = np.corrcoef(data_reconstracted[:,ff],data_real[:,ff])[0,1]
    for i in range(1000):
        if i%50==0: print(i)
        _ = np.zeros(32492)
        _[select_ind_L] = data_reconstracted[:, ff]
        eigen = SurfaceEigenstrapping(
                            data=_,
                            surface='/n02dat01/users/yfwang/Templates/surface/human/S1200.L.white_MSMAll.32k_fs_LR.surf.gii',
                            medial='/n01dat01/dyli/multi/support_code/BrainEigenmodes/data/template_surfaces_volumes/fsLR_32k_cortex-lh_mask.txt',
                            emodes=np.loadtxt('/n01dat01/dyli/multi/support_code/BrainEigenmodes/data/template_eigenmodes/fsLR_32k_white-lh_emode_200.txt'),
                            evals=np.loadtxt('/n01dat01/dyli/multi/support_code/BrainEigenmodes/data/template_eigenmodes/fsLR_32k_white-lh_eval_200.txt'),
                            num_modes=200,
                            resample=True,
                            )
        surr = eigen.generate()
        sur[i+1,ff] = np.corrcoef(surr[select_ind_L],data_real[:,ff])[0,1]
    np.save(f'/n01dat01/dyli/multi/results_data/RotationTest_Breakspear/{sub}_{label_name_lm[ff]}_SurfaceEigenstrapping_sur1000.npy', sur)

AF
0
IMPORTANT: EIGENMODES MUST BE TRUNCATED AT FIRST NON-ZERO MODE FOR THIS FUNCTION TO WORK
IMPORTANT: EIGENMODES MUST BE TRUNCATED AT FIRST NON-ZERO MODE FOR THIS FUNCTION TO WORK
IMPORTANT: EIGENMODES MUST BE TRUNCATED AT FIRST NON-ZERO MODE FOR THIS FUNCTION TO WORK
IMPORTANT: EIGENMODES MUST BE TRUNCATED AT FIRST NON-ZERO MODE FOR THIS FUNCTION TO WORK
IMPORTANT: EIGENMODES MUST BE TRUNCATED AT FIRST NON-ZERO MODE FOR THIS FUNCTION TO WORK
IMPORTANT: EIGENMODES MUST BE TRUNCATED AT FIRST NON-ZERO MODE FOR THIS FUNCTION TO WORK
IMPORTANT: EIGENMODES MUST BE TRUNCATED AT FIRST NON-ZERO MODE FOR THIS FUNCTION TO WORK
IMPORTANT: EIGENMODES MUST BE TRUNCATED AT FIRST NON-ZERO MODE FOR THIS FUNCTION TO WORK
IMPORTANT: EIGENMODES MUST BE TRUNCATED AT FIRST NON-ZERO MODE FOR THIS FUNCTION TO WORK
IMPORTANT: EIGENMODES MUST BE TRUNCATED AT FIRST NON-ZERO MODE FOR THIS FUNCTION TO WORK
IMPORTANT: EIGENMODES MUST BE TRUNCATED AT FIRST NON-ZERO MODE FOR THIS FUNCTION TO WORK
IMPORTANT: EIGEN

In [ ]:
from joblib import Parallel, delayed
import multiprocessing

In [ ]:
import numpy as np
from tqdm import tqdm
from eigenstrapping.rotations import rotate_matrix

M = np.loadtxt(f'/n01dat01/dyli/multi/support_code/BrainEigenmodes/data/template_eigenmodes/fsLR_32k_white-lh_emode_200.txt') # (32492, 200)
M_sur = np.zeros((32492, 200, 1000))
for i in tqdm(range(1000)):
    M_sur[...,i] = rotate_matrix(M, method='indirect', seed=None)
# np.save('/n01dat01/dyli/multi/results_data/RotationTest_Breakspear/fsLR_32k_white-lh_emode_200_Sur1000', M_sur)

In [ ]:
print('read sur modes')
M_sur = np.load('/n01dat01/dyli/multi/results_data/RotationTest_Breakspear/fsLR_32k_white-lh_emode_200_Sur1000.npy')
print('finish reading')

use rotation mode to reconstract tract reachability

In [ ]:
# remove the medial wall
dirc_L = '/n02dat01/users/dyli/Atlas/metric_index_L.txt'
select_ind_L = np.loadtxt( dirc_L ).astype(int)
dirc_R = '/n02dat01/users/dyli/Atlas/metric_index_R.txt'
select_ind_R = np.loadtxt( dirc_R ).astype(int)

_ = np.array([0, 2, 12, 14, 16, 18, 20, 22, 24, 26, 29, 31, 33, 35, 37, 39, 41, 43, 46, 48, 50, 52, 54, 56, 58, 60, 62, 64, 66, 68, 70, 4, 5, 6, 7, 8, 9, 10, 11, 28, 45])
# read the fiber name
l_idx = [1,3,13,15,17,19,25,27,30,32,36,38,40,42,44,47,49,51,53,55,57,59,61,63,65,67,69,71]
r_idx = [2,4,14,16,18,24,26,28,31,35,37,39,41,43,45,48,50,52,54,56,58,60,62,64,66,68,70,72]
m_idx = [4,5,6,7,8,9,10,11]
l_idx = np.array(l_idx)
r_idx = np.array(r_idx)
m_idx = np.array(m_idx)
l_idx = l_idx-1
r_idx = r_idx-1
label_f = open('/n02dat01/users/dyli/Grad_data/support_data/fiber_name_ori_nonum_nohemi.txt', 'r')
label_name = label_f.readlines()
label_name = [' '.join([i.strip() for i in price.strip().split('\n')]) for price in label_name]
label_name_lm = [label_name[l_idx[i]] for i in range(len(l_idx))] + [label_name[m_idx[i]] for i in range(len(m_idx))]
print(f'the number of fiber: {len(label_name_lm)}')

new_fiber_idx = []
for fi,ff in enumerate(_):
    if ff in list(l_idx)+list(m_idx): new_fiber_idx.append(fi)
new_fiber_idx = np.array(new_fiber_idx)
print(new_fiber_idx.shape)

print(len(l_idx), len(m_idx))

In [ ]:
# read the sublist
list_path = '/n02dat01/users/dyli/Grad_data/support_data/HCP_U100_list.txt'
with open( list_path, 'r' ) as f:
    namelist = [ str( line.strip()) for line in f.readlines() ]
print(f'the sub num is {len(namelist)}')

# read the group MODE
x = np.loadtxt(f'/n01dat01/dyli/multi/support_code/BrainEigenmodes/data/template_eigenmodes/fsLR_32k_white-lh_emode_200.txt') # (32492, 200)
x = x[select_ind_L,:]

In [ ]:
for sub in namelist[2:]:
    print(sub)
    if not os.path.exists(f'/n01dat01/dyli/multi/results_data/RotationTest_Breakspear/HCP_U100/FP_{sub}_predict_by_200_group_whitemode_deve-mode_thr05_para_L.npy'):
        # read the fingerprint
        if os.path.exists(f'/n04dat01/atlas_group/lma/HCP_S1200_individual_MSM_atlas/{sub}/{sub}_L_probtrackx_omatrix2/finger_print_fiber_MSMALL.npz'):
            _ = sps.load_npz(f'/n04dat01/atlas_group/lma/HCP_S1200_individual_MSM_atlas/{sub}/{sub}_L_probtrackx_omatrix2/finger_print_fiber_MSMALL.npz')
        _ = _.toarray()

        # check the all-zero row
        for ii in range(_.shape[0]):
            if len(np.unique(_[ii,:])) == 1: _[ii,:]=_[ii-1,:]
        for ii in range(_.shape[0]):
            if len(np.unique(_[ii,:])) == 1: sys.exit()

        # normalization
        fingerprint = np.array([_[:,i]/np.sum(_, axis=1) for i in range(_.shape[1])]).T

        # choose the left fibers and the cc fibers
        fingerprint = fingerprint[:, np.array(list(l_idx) + list(m_idx))]
        assert fingerprint.shape[0]==29696 and fingerprint.shape[1]==int(len(l_idx)+len(m_idx))

        # thr
        fingerprint[fingerprint<0.05] =0

        # for each item
        def para_calculate_my(i):
            x = np.squeeze(M_sur[...,i])
            x = x[select_ind_L,:]

            # corr_re = np.zeros(fingerprint.shape[1])
            para = np.zeros((200, fingerprint.shape[1]))
            for ff in range(fingerprint.shape[1]):
                y = fingerprint[:, ff]
                glm = sm.GLM(y,x, family=sm.families.Gaussian())
                glm_results = glm.fit()
                para[:,ff] = glm_results.params.T
                # corr_re[ff] = np.corrcoef(np.squeeze(y), np.squeeze(np.dot(x, glm_results.params.T)))[0,1]

            assert ~np.isnan(para).any()
            return para

        inputs = range(1000)
        # num_cores = multiprocessing.cpu_count()
        num_cores = 50
        print('the number of cores: ', num_cores)
        para_results = Parallel(n_jobs=num_cores)(delayed(para_calculate_my)(i) for i in inputs)
        para_results = np.squeeze(np.array(para_results))
        # corr_re_results = np.squeeze(np.array(corr_re_results))
        
        # save the correlation results for each sub
        np.save(f'/n01dat01/dyli/multi/results_data/RotationTest_Breakspear/HCP_U100/FP_{sub}_predict_by_200_group_whitemode_deve-mode_thr05_para_L.npy', para_results)
        # np.save(f'/n01dat01/dyli/multi/results_data/RotationTest_Breakspear/HCP_U100/FP_{sub}_predict_by_200_group_whitemode_deve-mode_thr05_pearsonr_L.npy', corr_re_results)
        print('-'*15,'Finished!!', '-'*15)

In [ ]:
for sub in ['101915']:
    print(sub)
    # read the fingerprint
    if os.path.exists(f'/n04dat01/atlas_group/lma/HCP_S1200_individual_MSM_atlas/{sub}/{sub}_L_probtrackx_omatrix2/finger_print_fiber_MSMALL.npz'):
        _ = sps.load_npz(f'/n04dat01/atlas_group/lma/HCP_S1200_individual_MSM_atlas/{sub}/{sub}_L_probtrackx_omatrix2/finger_print_fiber_MSMALL.npz')
    _ = _.toarray()

    # check the all-zero row
    for ii in range(_.shape[0]):
        if len(np.unique(_[ii,:])) == 1: _[ii,:]=_[ii-1,:]
    for ii in range(_.shape[0]):
        if len(np.unique(_[ii,:])) == 1: sys.exit()

    # normalization
    fingerprint = np.array([_[:,i]/np.sum(_, axis=1) for i in range(_.shape[1])]).T

    # choose the left fibers and the cc fibers
    fingerprint = fingerprint[:, np.array(list(l_idx) + list(m_idx))]
    assert fingerprint.shape[0]==29696 and fingerprint.shape[1]==int(len(l_idx)+len(m_idx))

    # thr
    fingerprint[fingerprint<0.05] =0

    para_results = np.load(f'/n01dat01/dyli/multi/results_data/RotationTest_Breakspear/HCP_U100/FP_{sub}_predict_by_200_group_whitemode_deve-mode_thr05_para_L.npy') # (10, 200, 36)
    print(para_results.shape)

    corr_re = np.zeros((fingerprint.shape[1],1000))
    for i in range(1000):
        para = np.squeeze(para_results[i,...])
        for ff in range(36):
            y = fingerprint[:, ff]
            corr_re[ff,i] = np.corrcoef(np.squeeze(y), np.squeeze(np.dot(x, para[:,ff])))[0,1]
    np.save(f'/n01dat01/dyli/multi/results_data/RotationTest_Breakspear/HCP_U100/FP_{sub}_predict_by_200_group_whitemode_deve-mode_thr05_pearsonr_L.npy', corr_re)
    sns.heatmap(corr_re,cmap='coolwarm')
    plt.title(sub)
    plt.show()

In [ ]:
corr_true = np.load('/n01dat01/dyli/multi/HCP_1200/101915/FP_101915_predict_by_500_group_whitemode_deve-mode_thr05_pearsonr_L.npy')[new_fiber_idx]
corr_true.shape

In [ ]:
p = np.array([len(np.argwhere(corr_re[i,:]>corr_true[i]))/1000 for i in range(36)])
p